# الفصل 29: إعادة استخدام النماذج: إطار القرار الشامل

> بعد فصول 24-28، امتلكت خمس تقنيات قوية مختلفة (CPT، Distillation، Merging، Pruning، Upscaling).

- **الجزء:** 6 — إعادة استخدام النماذج
- **الفصل:** 29
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 28: توسيع العمق: Depth Upscaling
- **الوسوم:** Model Reuse، Decision Framework، Trade-offs، Case Studies

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch29-model-reuse-framework.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-06-model-reuse%2Fch29-model-reuse-framework.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-06-model-reuse/ch29-model-reuse-framework.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch29-model-reuse-framework.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch29-model-reuse-framework.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch29-model-reuse-framework.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** لا توجد مكتبات خارجية مطلوبة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [

]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch29-model-reuse-framework.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-06-model-reuse%2Fch29-model-reuse-framework.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-06-model-reuse/ch29-model-reuse-framework.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch29-model-reuse-framework.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch29-model-reuse-framework.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch29-model-reuse-framework.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> بعد فصول 24-28، امتلكت خمس تقنيات قوية مختلفة (CPT، Distillation، Merging، Pruning، Upscaling).

> **بيانات الفصل**
>
> - **الجزء:** 6 — إعادة استخدام النماذج
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 28: توسيع العمق: Depth Upscaling


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم Model Reuse
- فهم Decision Framework
- فهم Trade-offs


## 1. لماذا يهمّك هذا الموضوع؟

🟢 طبقة الحدس
بعد فصول 24-28، امتلكت خمس تقنيات قوية مختلفة (CPT، Distillation، Merging، Pruning، Upscaling). السؤال العملي الحقيقي الآن: كيف تختار التقنية (أو مزيج التقنيات) الصحيحة لمشكلتك المحددة؟ هذا الفصل يُقدِّم إطار قرار منهجيًا بدل الاختيار العشوائي أو التجربة والخطأ غير الموجَّهة.


## 2. الخريطة الذهنية


```text
ابدأ هنا: ما هو هدفك الأساسي؟
│
├── "أريد نموذجًا أصغر وأسرع للإنتاج (نفس القدرة تقريبًا)"
│   │
│   ├── هل تحتاج تسريعًا فعليًا على أي هاردوير؟
│   │   ├── نعم → Structured Pruning (الفصل 27) + Recovery Fine-tuning
│   │   └── لا، فقط توفير ذاكرة → Unstructured Pruning (Wanda/SparseGPT)
│   │
│   └── هل تحتاج تصغيرًا جذريًا (10× أو أكثر)؟
│       └── نعم → Knowledge Distillation (الفصل 25) لنموذج تلميذ جديد
│                  أصغر بنيويًا، وليس مجرد تقليم النموذج الحالي
│
├── "أريد نموذجًا يعرف مجالي/لغتي بعمق أكبر"
│   │
│   ├── الفجوة معرفية عميقة (مفاهيم/مفردات جديدة كليًا)
│   │   └── Continued Pretraining (الفصل 24) على نصوص المجال الخام
│   │
│   └── الفجوة سلوكية فقط (يعرف لكن لا يتبع التنسيق المطلوب)
│       └── انتقل مباشرة لـ SFT/DPO (الجزء السابع) - لا حاجة لـ CPT
│
├── "أملك عدة نماذج مُضبَّطة لمهام مختلفة وأريد دمج قدراتها"
│   │
│   ├── النماذج من نفس الأساس تمامًا، مهام متعددة
│   │   └── TIES-Merging + DARE (الفصل 26) - الخيار الافتراضي القوي
│   │
│   └── نموذجان فقط، متقاربان جدًا في الأداء
│       └── SLERP (الفصل 26) كافٍ وأبسط
│
└── "أريد نموذجًا أقوى (أكبر سعة) دون تدريب من الصفر"
    │
    └── يوجد نموذج أساسي قوي بنفس المعمارية تقريبًا
        └── Depth Upscaling (الفصل 28) + CPT قصير
```


## 3. الشرح المتدرج

الإطار ليس شجرة قرار جامدة، بل **عملية من ست خطوات** تُطبَّق قبل اختيار أي تقنية:

1. **حدّد الفجوة بدقة.** هل المشكلة معرفة ناقصة (المجال/اللغة)، أم سلوك غير مرغوب، أم قيود حجم/تكلفة، أم نقص بيانات؟
2. **قِس خط الأساس.** ما أداء النموذج الحالي على مجموعة تقييم تمثيلية؟ بدون خط أساس لا يمكن معرفة هل التحسين يستحق التكلفة.
3. **قدّر الميزانية.** كم GPU-ساعة، وكم أسبوع، وكم بيانات مُعلَّمة/خام متاحة؟
4. **افحص البدائل الأرخص أولًا.** Prompting المتقدم، RAG، إعادة ترتيب، أو ضبط SFT خفيف قد تحل المشكلة بتكلفة أقل بكثير من CPT أو الدمج.
5. **اختر أقصر مسار يحقق معيار القبول.** استخدم الشجرة في القسم التالي، ثم نفّذ تجربة مصغّرة (Pilot) على نموذج صغير قبل الالتزام الكامل.
6. **قِس التكلفة الكلية.** ليست تكلفة التدريب فقط، بل الاستدلال، والصيانة، وإعادة التقييم، ومخاطر التراجع.

**مصفوفة قرار مبسّطة:**

| السؤال | إذا كانت الإجابة «نعم» | التقنية المرشحة |
|---|---|---|
| هل ينقص النموذج معرفة مجال/لغة؟ | نعم | CPT أو DAPT |
| هل يعرف النموذج المعلومة لكنه لا يتبع التعليمات؟ | نعم | SFT / LoRA |
| هل الأسلوب صحيح لكن التفضيلات غير مرغوبة؟ | نعم | DPO / RLHF |
| هل تحتاج نموذجًا أصغر/أسرع؟ | نعم | Distillation أو Pruning |
| هل لديك عدة نماذج متخصصة وتريد واحدًا؟ | نعم | Model Merging |
| هل تريد قدرة أكبر بميزانية محدودة؟ | نعم | Depth Upscaling |
| هل يمكن حل المشكلة دون تدريب؟ | نعم | Prompting / RAG |


## 4. مثال محلول خطوة بخطوة

### 💡 دراسات حالة من الصناعة


```text
مثال 1 - استراتيجية بناء نموذج حافة إنتاجي (مشابه لممارسات شائعة):
  نموذج معلّم كبير (70B)
    → Reasoning Distillation (الفصل 25) إلى نموذج تلميذ 7B
    → Structured Pruning (الفصل 27) لتصغيره إلى ~5B
    → Recovery Fine-tuning (LoRA قصير)
    → BitNet/Quantization (الفصل 63) للنشر على الحافة

مثال 2 - استراتيجية نموذج متخصص بمجال ولغة (كما في الجزء المشروع الختامي أدناه):
  نموذج أساسي عام (Llama 3 8B)
    → Continued Pretraining (الفصل 24) على نصوص عربية طبية
    → عدة نسخ SFT/LoRA متخصصة (تشخيص، صيدلة، تمريض)
    → TIES-Merging (الفصل 26) لدمج الثلاث تخصصات في نموذج واحد
    → تقييم شامل (الجزء التاسع) قبل النشر

مثال 3 - استراتيجية SOLAR الفعلية (موثّقة):
  Mistral 7B (مُدرَّب مسبقًا)
    → Depth Upscaling (الفصل 28) إلى 10.7B
    → Continued Pretraining قصير
    → SFT + DPO (الجزء السابع) → SOLAR-10.7B-Instruct
```


## 5. نصائح وحيل

- **لا تبدأ بالتقنية.** ابدأ بتعريف المشكلة ومعيار النجاح القابل للقياس.
- **جرّب الأرخص أولًا**: RAG أو Prompting قد يغنيان عن أي تدريب.
- **أجرِ تجربة مصغّرة** على نموذج 0.5B–1B قبل الالتزام بنموذج 7B+.
- **احسب التكلفة الكلية** (تدريب + استدلال + تقييم + صيانة)، لا تكلفة التدريب فقط.
- **حدّد معيار قبول مسبقًا** (مثل: +5 نقاط F1 دون انخفاض >1 في MMLU) لتفادي قرارات عاطفية.
- **وثّق كل تجربة** (البيانات، الـhyperparameters، النتائج) لبناء ذاكرة مؤسسية.
- **خطّط للتراجع**: احتفظ بالنموذج السابق ومسار إعادة النشر إذا فشل النموذج الجديد.
- **أعد التقييم بعد النشر**؛ الأداء في المختبر لا يساوي الأداء على بيانات المستخدمين.


## 6. أخطاء شائعة ومصائد

- **اختيار التقنية قبل تعريف المشكلة**: «لنجرّب LoRA» ليس هدفًا هندسيًا.
- **تجاهل جودة البيانات**: CPT على بيانات رديئة يُنتج نموذجًا واثقًا في الخطأ.
- **الاستهانة بتكلفة التقييم**: بدون مجموعة تقييم موثوقة، لا يمكن معرفة هل التحسين حقيقي.
- **دمج نماذج بأساس مختلف**: يفشل غالبًا حتى لو تطابقت الأشكال ظاهريًا.
- **تقليم بنيوي دون تعافٍ**: النموذج الناتج ينهار في المهام المعقدة.
- **CPT دون Replay**: نسيان كارثي للقدرات العامة.
- **مقارنة تقنيات بتكلفة مختلفة** دون تطبيع النتائج (GPU-ساعة/نقطة أداء).
- **نسيان الاستدلال**: نموذج أفضل بـ2% لكنه أبطأ 3× قد يكون أسوأ للمنتج.


## 7. جدول مقارنة

| التقنية | الهدف الأساسي | التكلفة النسبية | يحتاج بيانات تدريب؟ | التأثير على الحجم |
|---|---|---|---|---|
| Continued Pretraining | حقن معرفة/لغة جديدة | متوسطة (مليارات رموز) | نعم (نص خام غير مُعلَّم) | لا يتغيّر |
| Knowledge Distillation | ضغط لنموذج أصغر بنيويًا | متوسطة-عالية (توليد + تدريب) | نعم (استجابات المُعلِّم) | يصغر جذريًا |
| Model Merging | دمج قدرات متعددة بلا تدريب | منخفضة جدًا (عمليات حسابية فقط) | لا (في أبسط الحالات) | يبقى كما هو |
| Pruning | تصغير وتسريع نموذج موجود | منخفضة-متوسطة (+ تعافٍ خفيف) | نعم (معايرة + تعافٍ) | يصغر جزئيًا |
| Depth Upscaling | تكبير السعة دون تدريب كامل | منخفضة-متوسطة (CPT قصير) | نعم (CPT) | يكبر |


## 8. ملخص أفضل الممارسات

| السؤال | إذا كانت الإجابة "نعم" |
|---|---|
| هل مشكلتي الأساسية هي حجم/سرعة النموذج في الإنتاج؟ | ابدأ بـ Pruning أو Distillation |
| هل مشكلتي الأساسية هي نقص معرفة بمجال/لغة محددة؟ | ابدأ بـ Continued Pretraining |
| هل أملك عدة نماذج متخصصة وأريد واحدًا شاملاً؟ | ابدأ بـ Model Merging (TIES+DARE) |
| هل أملك ميزانية حوسبة محدودة جدًا وأريد نموذجًا أقوى؟ | فكّر في Depth Upscaling (إن وُجد أساس مناسب) |
| هل جرّبت الحل الأبسط (Prompting/RAG من الجزء العاشر) قبل اللجوء لإعادة تدريب النموذج أصلاً؟ | تحقق من هذا دائمًا أولاً — قد لا تحتاج أيًا من هذه التقنيات المكلفة نسبيًا |

> ⚠️ تذكير ختامي مهم: قبل الاستثمار في أي من تقنيات هذا الجزء، تأكد دائمًا أن المشكلة لا يمكن حلها بشكل أبسط وأرخص عبر هندسة الأوامر المتقدمة أو RAG (الجزء العاشر) — إعادة استخدام/تعديل النموذج نفسه خطوة أكبر تكلفة ومخاطرة (قد تُدخِل نسيانًا كارثيًا أو تدهورًا غير متوقع) ويجب تبريرها بوضوح مقابل البدائل الأبسط.


## 9. أسئلة مقابلات

**س: كيف تختار بين CPT وSFT عندما يكون لديك بيانات مجال جديدة؟**

ج: اسأل: هل البيانات تُعلّم **معرفة** أم **سلوكًا**؟ إذا كانت نصوصًا خامًا تُعرّف النموذج بمصطلحات/حقائق/أسلوب المجال، فالأرجح CPT (مع Replay). إذا كانت أزواج (تعليمات، إجابات) تُعلّم النموذج كيفية الاستجابة، فـSFT كافٍ. كثير من المشاريع تحتاج الاثنين: CPT أولًا ثم SFT.

**س: متى يكون Model Merging أفضل من تدريب نموذج واحد على كل المهام؟**

ج: عندما تملك بالفعل عدة نماذج متخصصة من نفس الأساس، وتريد دمج قدراتها دون تكلفة تدريب جديدة. يكون أفضل عندما تكون النماذج متقاربة (نفس المعمارية والتوكنizer) وتستخدم TIES/DARE لتقليل التداخل. أما إذا كانت المهام متعارضة بشدة أو البيانات مشتركة، فقد يتفوق التدريب المشترك.

**س: كيف تقرر بين Pruning وDistillation لتقليل حجم النموذج؟**

ج: Pruning يحافظ على المعمارية ويقلل الأوزان/المكوّنات، وهو أرخص لكنه محدود بنسبة تقليم آمنة ويحتاج تعافيًا. Distillation يدرّب نموذجًا أصغر من الصفر (أو من نموذج أصغر) على مخرجات المعلّم، ويعطي عادة جودة أفضل عند نسب تصغير كبيرة، لكنه أغلى. القاعدة: جرّب Pruning أولًا إذا كان التصغير ≤30–40%، وDistillation إذا احتجت تصغيرًا أكبر أو معمارية مختلفة.


## 10. تمرين عملي

اختر سيناريو واقعيًا (مثل: «نموذج عربي عام 7B يحتاج فهمًا أفضل للمصطلحات القانونية السعودية، وميزانيتك 4 GPU-ساعات فقط»)، ثم:

1. طبّق الخطوات الست في القسم 3.
2. املأ مصفوفة القرار.
3. حدّد التقنية (أو مزيجًا) مع تبرير.
4. صمّم تجربة مصغّرة: ما النموذج؟ ما البيانات؟ ما معيار القبول؟
5. قدّر التكلفة الكلية بالدولار.
6. اكتب فقرة «متى نتراجع؟».

شارك إجابتك مع زميل واطلب منه مهاجمتها: هل كان بإمكانك حل المشكلة بـRAG أو Prompting؟


## 11. مراجع للتعمّق

- Gururangan et al. (2020): *Don't Stop Pretraining: Adapt Language Models to Domains and Tasks*.
- Hinton et al. (2015): *Distilling the Knowledge in a Neural Network*.
- Wortsman et al. (2022): *Model Soups*.
- Yadav et al. (2023): *TIES-Merging*.
- Frantar & Alistarh (2023): *SparseGPT*.
- Kim et al. (2023): *SOLAR 10.7B*.
- Hugging Face PEFT documentation — https://huggingface.co/docs/peft
- mergekit — https://github.com/arcee-ai/mergekit


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — إعادة استخدام النماذج / الفصل 29: إعادة استخدام النماذج: إطار القرار الشامل.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
